# HW12: временные ряды, temporal split, GRU

Файл `S12-hw-dataset.csv` в этой папке.

In [1]:
import os, json, csv, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve()
if ROOT.name != "HW12":
    cand = ROOT / "homeworks" / "HW12"
    if cand.is_dir():
        os.chdir(cand)
ART = Path("artifacts")
FIG = ART / "figures"
FIG.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

csv_path = Path("S12-hw-dataset.csv")
df = pd.read_csv(csv_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
print(df.head(), len(df))

# temporal split 70/15/15
n = len(df)
n_test = int(n * 0.15)
n_val = int(n * 0.15)
n_train = n - n_val - n_test
tr = df.iloc[:n_train].copy()
va = df.iloc[n_train : n_train + n_val].copy()
te = df.iloc[n_train + n_val :].copy()
print("split", n_train, len(va), len(te))

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(tr["date"], tr["target"], label="train")
ax.plot(va["date"], va["target"], label="val")
ax.plot(te["date"], te["target"], label="test")
ax.legend()
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(FIG / "series_split.png", dpi=120)
plt.close()

# features without leakage
def add_features(d):
    d = d.copy()
    d["lag_1"] = d["target"].shift(1)
    d["lag_7"] = d["target"].shift(7)
    d["lag_14"] = d["target"].shift(14)
    d["rolling_mean_7"] = d["target"].shift(1).rolling(7).mean()
    d["rolling_std_7"] = d["target"].shift(1).rolling(7).std()
    d["dow"] = d["date"].dt.dayofweek
    return d

full = add_features(df)
tr_f = full.iloc[:n_train].dropna()
va_f = full.iloc[n_train : n_train + n_val].dropna()
te_f = full.iloc[n_train + n_val :].dropna()
feat_cols = ["lag_1", "lag_7", "lag_14", "rolling_mean_7", "rolling_std_7", "dow"]
Xtr, ytr = tr_f[feat_cols].values, tr_f["target"].values
Xva, yva = va_f[feat_cols].values, va_f["target"].values
Xte, yte = te_f[feat_cols].values, te_f["target"].values

scaler = StandardScaler()
Xtr_s = scaler.fit_transform(Xtr)
Xva_s = scaler.transform(Xva)
Xte_s = scaler.transform(Xte)

def metrics(y, yhat):
    mae = np.mean(np.abs(y - yhat))
    rmse = math.sqrt(np.mean((y - yhat) ** 2))
    mape = np.mean(np.abs((y - yhat) / (np.abs(y) + 1e-8))) * 100
    return mae, rmse, mape

# B1 naive last
y_va_hat_b1 = va_f["lag_1"].values
y_te_hat_b1 = te_f["lag_1"].values
b1_va = metrics(yva, y_va_hat_b1)
b1_te = metrics(yte, y_te_hat_b1)

# B2 moving average window 7 on shifted series
def ma_predict(series_vals, window):
    out = []
    for i in range(len(series_vals)):
        if i < window:
            out.append(series_vals[max(0, i - 1)] if i > 0 else series_vals[0])
        else:
            out.append(np.mean(series_vals[i - window : i]))
    return np.array(out)

# align: use train statistics for val/test — apply on full target series positions
full_target = df["target"].values
# simpler: val predictions = lag_1 as proxy OR compute rolling mean from past only in loop
window = 24
y_va_b2 = np.array([full_target[n_train + i - 1] if i == 0 else np.mean(full_target[n_train + i - window : n_train + i]) for i in range(len(va))])
# fix: use explicit loop from history
hist = full_target[: n_train]
va_preds = []
for i in range(len(va)):
    idx = n_train + i
    w = full_target[max(0, idx - window) : idx]
    va_preds.append(np.mean(w) if len(w) else full_target[idx - 1])
y_va_b2 = np.array(va_preds)
y_te_b2 = []
for i in range(len(te)):
    idx = n_train + n_val + i
    w = full_target[max(0, idx - window) : idx]
    y_te_b2.append(np.mean(w) if len(w) else full_target[idx - 1])
y_te_b2 = np.array(y_te_b2)
b2_va = metrics(yva, y_va_b2)
b2_te = metrics(yte, y_te_b2)

# B3 Ridge
ridge = Ridge(alpha=1.0, random_state=SEED)
ridge.fit(Xtr_s, ytr)
y_va_b3 = ridge.predict(Xva_s)
y_te_b3 = ridge.predict(Xte_s)
b3_va = metrics(yva, y_va_b3)
b3_te = metrics(yte, y_te_b3)

# R1 GRU window
WINDOW = 48
HIDDEN = 64
series = df["target"].values.astype(np.float32)

class SeqDS(Dataset):
    def __init__(self, arr, start, end):
        self.data = arr[start:end]
    def __len__(self):
        return max(0, len(self.data) - WINDOW - 1)
    def __getitem__(self, i):
        x = self.data[i : i + WINDOW]
        y = self.data[i + WINDOW]
        return torch.from_numpy(x).unsqueeze(-1), torch.tensor(y)

# scale series for GRU
mu, sig = series[:n_train].mean(), series[:n_train].std() + 1e-8
series_n = (series - mu) / sig

tr_ds = SeqDS(series_n, 0, n_train)
va_ds = SeqDS(series_n, 0, n_train + n_val)
# validation windows that fall inside val region only
class ValDS(Dataset):
    def __init__(self):
        pass
    def __len__(self):
        return n_val - WINDOW - 1
    def __getitem__(self, i):
        idx = n_train + i
        x = series_n[idx - WINDOW : idx]
        y = series_n[idx]
        return torch.from_numpy(x).unsqueeze(-1), torch.tensor(y)

va_gru = ValDS()
tr_loader = DataLoader(tr_ds, batch_size=64, shuffle=True)
va_loader = DataLoader(va_gru, batch_size=128, shuffle=False)

class GRUModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(1, HIDDEN, batch_first=True)
        self.fc = nn.Linear(HIDDEN, 1)
    def forward(self, x):
        o, _ = self.gru(x)
        return self.fc(o[:, -1, :]).squeeze(-1)

def train_gru():
    m = GRUModel().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    crit = nn.MSELoss()
    best = 1e9
    hist = {"tl": [], "vl": []}
    for ep in range(40):
        m.train()
        tl = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = m(xb)
            loss = crit(pred, yb)
            loss.backward()
            opt.step()
            tl += loss.item() * xb.size(0)
        tl /= len(tr_loader.dataset)
        m.eval()
        vl = 0.0
        with torch.no_grad():
            for xb, yb in va_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = m(xb)
                vl += crit(pred, yb).item() * xb.size(0)
        vl /= max(1, len(va_loader.dataset))
        hist["tl"].append(tl)
        hist["vl"].append(vl)
        if vl < best:
            best = vl
            torch.save(m.state_dict(), ART / "best_gru.pt")
        if ep % 10 == 0:
            print("ep", ep, "val mse", vl)
    return hist

h_gru = train_gru()
m = GRUModel().to(device)
m.load_state_dict(torch.load(ART / "best_gru.pt", map_location=device))

def gru_predict_on(start, end):
    preds = []
    for idx in range(start + WINDOW, end):
        x = torch.from_numpy(series_n[idx - WINDOW : idx]).float().unsqueeze(0).unsqueeze(-1).to(device)
        with torch.no_grad():
            p = m(x).item()
        preds.append(p * sig + mu)
    return np.array(preds)

y_va_r = gru_predict_on(n_train, n_train + n_val)
y_va_true = series[n_train + WINDOW : n_train + n_val]
b_r_va = metrics(y_va_true, y_va_r)

y_te_r = gru_predict_on(n_train + n_val, n)
y_te_true = series[n_train + n_val + WINDOW :]
b_r_te = metrics(y_te_true, y_te_r)

# choose best by val MAE among B1,B2,B3,R1
cands = {
    "B1": (b1_va[0], "naive-last"),
    "B2": (b2_va[0], "ma"),
    "B3": (b3_va[0], "ridge"),
    "R1": (b_r_va[0], "gru"),
}
best_id = min(cands, key=lambda k: cands[k][0])
print("best by val MAE", best_id, cands[best_id])

fig, ax = plt.subplots()
ax.plot(["B1", "B2", "B3", "R1"], [b1_va[0], b2_va[0], b3_va[0], b_r_va[0]], marker="o")
ax.set_ylabel("Val MAE")
plt.title("Baselines vs GRU")
plt.tight_layout()
plt.savefig(FIG / "baselines_compare.png", dpi=120)
plt.close()

fig, ax = plt.subplots()
ax.plot(h_gru["tl"], label="train")
ax.plot(h_gru["vl"], label="val")
ax.legend()
plt.savefig(FIG / "gru_learning_curves.png", dpi=120)
plt.close()

# best forecast on test — use best model logic
if best_id == "R1":
    y_hat_plot = y_te_r
    y_true_plot = y_te_true
else:
    y_hat_plot = {"B1": y_te_hat_b1, "B2": y_te_b2, "B3": y_te_b3}[best_id]
    y_true_plot = yte[-len(y_hat_plot):] if len(y_hat_plot) != len(yte) else yte

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(y_true_plot[:500], label="true")
ax.plot(y_hat_plot[:500], label="pred")
ax.legend()
plt.savefig(FIG / "best_forecast_test.png", dpi=120)
plt.close()

rows = [
    {"experiment_id": "B1", "task": "forecasting", "dataset": "S12-hw-dataset.csv", "seed": SEED,
     "split_summary": "70/15/15 time", "window_size": "", "horizon": 1,
     "model_summary": "naive last", "features_summary": "lag_1", "scaler": "",
     "optimizer": "", "lr": "", "epochs_trained": "", "best_val_mae": b1_va[0], "best_val_rmse": b1_va[1], "best_val_mape": b1_va[2],
     "test_mae": b1_te[0], "test_rmse": b1_te[1], "test_mape": b1_te[2], "notes": ""},
    {"experiment_id": "B2", "task": "forecasting", "dataset": "S12-hw-dataset.csv", "seed": SEED,
     "split_summary": "70/15/15 time", "window_size": window, "horizon": 1,
     "model_summary": f"MA({window})", "features_summary": "past window mean", "scaler": "",
     "optimizer": "", "lr": "", "epochs_trained": "", "best_val_mae": b2_va[0], "best_val_rmse": b2_va[1], "best_val_mape": b2_va[2],
     "test_mae": b2_te[0], "test_rmse": b2_te[1], "test_mape": b2_te[2], "notes": ""},
    {"experiment_id": "B3", "task": "forecasting", "dataset": "S12-hw-dataset.csv", "seed": SEED,
     "split_summary": "70/15/15 time", "window_size": "", "horizon": 1,
     "model_summary": "Ridge", "features_summary": ",".join(feat_cols), "scaler": "StandardScaler train",
     "optimizer": "", "lr": "", "epochs_trained": "", "best_val_mae": b3_va[0], "best_val_rmse": b3_va[1], "best_val_mape": b3_va[2],
     "test_mae": b3_te[0], "test_rmse": b3_te[1], "test_mape": b3_te[2], "notes": ""},
    {"experiment_id": "R1", "task": "forecasting", "dataset": "S12-hw-dataset.csv", "seed": SEED,
     "split_summary": "70/15/15 time", "window_size": WINDOW, "horizon": 1,
     "model_summary": f"GRU hidden={HIDDEN}", "features_summary": "scaled target window", "scaler": "z-score train",
     "optimizer": "Adam", "lr": 1e-3, "epochs_trained": 40, "best_val_mae": b_r_va[0], "best_val_rmse": b_r_va[1], "best_val_mape": b_r_va[2],
     "test_mae": b_r_te[0], "test_rmse": b_r_te[1], "test_mape": b_r_te[2], "notes": ""},
]
with open(ART / "runs.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

with open(ART / "best_gru_config.json", "w", encoding="utf-8") as f:
    json.dump({"window_size": WINDOW, "hidden": HIDDEN, "seed": SEED, "epochs": 40}, f, indent=2)

print("HW12 done")

                 date  target
0 2025-01-01 00:00:00   98.14
1 2025-01-01 01:00:00   98.07
2 2025-01-01 02:00:00  104.70
3 2025-01-01 03:00:00  112.81
4 2025-01-01 04:00:00  112.62 4320
split 3024 648 648
ep 0 val mse 0.6694355578175769
ep 10 val mse 0.15153122786587983
ep 20 val mse 0.150797675816165
ep 30 val mse 0.14701080197881974
best by val MAE R1 (np.float32(5.0496917), 'gru')
HW12 done
